# Do the mouse AKI calls hold in human AKI? A KPMP benchmark (notebook 98)

**The question.** Notebook 94 scored our mouse AKI call lists on another lab's mouse IRI. Which positional injury
changes of the mouse PT also appear in human acute kidney injury? The KPMP/HuBMAP atlas (Lake et al. 2023) has PT
nuclei from AKI biopsies and reference tissue. This notebook scores the same call lists and claim checks on the human
contrast, adds notebook 95's signalling finding (PROGENy EGFR, MAPK, PI3K), and sets human beside mouse.

**The frozen protocol** is `docs/results/pt-human-aki-benchmark.md` (commit 1727af4). Scoring follows notebook 94
exactly, with two contrasts in place of time points: the primary (AKI against all reference tissue) and the
sensitivity (AKI against healthy reference tissue only), which also serves as the stability check.

**Sections.** 1 setup and donors; 2 pseudobulks; 3 human gene statistics; 4 our call lists; 5 set statistics;
6 agreement of levels; 7 scores and comparisons; 8 threshold-free agreement; 9 claim checks and K10; 10 figures;
11 reading and run record.

## 1 · Setup and donors

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
KPMP = data_root / 'external' / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
NETS = data_root / 'external' / 'regulator_networks'
STATE = data_root / 'external' / 'kidney_physiological_state'
XIONG = data_root / 'external' / 'xiong2023_sex_bias'
ORTHOLOGS = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
T71 = results_root / 'pt_pathways_package_pseudospace'
T76 = results_root / 'pt_package_aki' / 'tables'
T80 = results_root / 'pt_known_injury_programs' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T93 = results_root / 'pt_pathway_networks' / 'tables'
T94 = results_root / 'pt_external_iri' / 'tables'
output = results_root / 'pt_human_aki_benchmark'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '98.human_aki.1'
SEED = 98
TIMES = ['primary', 'healthy only']            # contrasts, in the role of notebook 94's time points
ACUTE = TIMES                                  # a comparison is stable when it also holds in the other contrast
T_LABEL = {'primary': 'primary', 'healthy only': 'healthy only'}
T_STAR = 'primary'
REFERENCE = {'primary': ['Healthy_reference_tissue', 'Healthy_stone_donor', 'Tumor_nephrectomy'],
             'healthy only': ['Healthy_reference_tissue']}
MIN_NUCLEI, MIN_SEGMENT = 100, 20
FROZEN_CALLS = {'F71': 103, 'C71': 459, 'N93': 134, 'F76': 106, 'level genes': 3159, 'location genes': 362,
                'level sets': 538, 'location sets': 236}
ALPHA = .05
MIN_CPM, MIN_DONORS = 1., 3
SET_SIZES = (10, 300)
MIN_PROGRAM = 5
CAMERA_COR = .01
N_NULL = 1000
LATE = 1.8
MIN_N_COMPARE, MIN_N_REPLICATE, MARGIN = 20, 10, .10
MT_PREFIX = 'mt-'
TOD_LFC, TOD_P, FAST_LFC, FAST_CPM = .5, .01, 1., 10.
PROGENY_TOP, N_PERM = 500, 1000
REQUIRED = [KPMP, ORTHOLOGS, NETS / 'progeny_human.tsv', T94 / 'list_scores.csv', T94 / 'claim_checks.csv',
            T71 / 'tables' / 'universe_aki_vs_control.csv', T76 / 'universe.csv', T91 / 'i_genes.csv.gz',
            T91 / 'i_induced_genes.csv', T91 / 'i_lost_genes.csv', T93 / 'calls_aki_vs_control.csv',
            T80 / 'programs.csv', T80 / 'programs_members.csv', STATE / 'GSE267280_RNA_Seq_counts.txt.gz']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 71, 76, 80, 91, 93 and 94 first)')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.stats import norm, spearmanr
from IPython.display import display

from pseudospace.external_replication import log_cpm, matched_null, null_summary, replicated
from pseudospace.species_level import limma_trend_species

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
LIST_COLOR = {'G-L91': '#2a78d6', 'G-M91': '#9C93D3', 'G-F71': '#1b8a5a', 'G-C71': '#eda100',
              'S-F71': '#1b8a5a', 'S-F76': '#5fb88a', 'S-C71': '#eda100', 'S-N93': '#d55e00',
              'S-L91': '#2a78d6', 'S-M91': '#9C93D3', 'S-P80': '#52514e'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
kp = ad.read_h5ad(KPMP, backed='r')
OBS = kp.obs[['donor_id', 'disease_category', 'region', 'subclass.l1', 'subclass.l2', 'sex', 'Age_binned']].copy()
HUMAN_GENES = pd.Index(kp.var['feature_name'].astype(str))
del kp
with h5py.File(KPMP, 'r') as f:
    assert int(f['raw/X'].attrs['shape'][1]) == len(HUMAN_GENES), 'raw/X and var disagree on genes'
for c in OBS.columns:
    OBS[c] = OBS[c].astype(str)
keep = OBS['subclass.l1'].eq('PT') & ~OBS['subclass.l2'].eq('dPT/DTL') & OBS.region.eq('C')
cats = ['AKI'] + REFERENCE['primary']
keep &= OBS.disease_category.isin(cats)
counts_by_donor = OBS[keep].groupby('donor_id').size()
DONORS = counts_by_donor.index[counts_by_donor >= MIN_NUCLEI].tolist()
keep &= OBS.donor_id.isin(DONORS)
PTN = OBS[keep].copy()
DONOR_TABLE = PTN.groupby('donor_id').agg(category=('disease_category', 'first'), sex=('sex', 'first'),
                                          age=('Age_binned', 'first'), nuclei=('donor_id', 'size'))
for s in ['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']:
    DONOR_TABLE[s] = PTN[PTN['subclass.l2'].eq(s)].groupby('donor_id').size().reindex(DONOR_TABLE.index).fillna(0).astype(int)
DONOR_TABLE.to_csv(tables / 'donors.csv')
display(DONOR_TABLE)
print(DONOR_TABLE.groupby('category').agg(donors=('nuclei', 'size'), nuclei=('nuclei', 'sum')).to_string())
assert DONOR_TABLE.category.eq('AKI').sum() >= 10, 'too few AKI donors'
guard_log.append(f"donors: AKI {int(DONOR_TABLE.category.eq('AKI').sum())}, reference {int((~DONOR_TABLE.category.eq('AKI')).sum())} "
                 f"(≥ {MIN_NUCLEI} cortical PT nuclei)")

## 2 · Pseudobulks

Raw counts streamed from `raw/X` in row chunks; per donor × PT subclass sums over every human gene.

In [ ]:
def h5_group_sums(path, rows, labels, *, chunk=20000):
    """Per-label sums of the CSR rows of raw/X (genes x labels), streamed in row chunks."""
    lab = pd.Index(sorted(set(labels)))
    code = pd.Series(lab.get_indexer(labels), index=rows)
    with h5py.File(path, 'r') as f:
        X = f['raw/X']
        indptr = X['indptr'][:]
        n_rows, n_genes = X.attrs['shape']
        out = np.zeros((n_genes, len(lab)))
        for start in range(0, n_rows, chunk):
            stop = min(start + chunk, n_rows)
            want = code[(code.index >= start) & (code.index < stop)]
            if want.empty:
                continue
            lo, hi = indptr[start], indptr[stop]
            block = sparse.csr_matrix((X['data'][lo:hi], X['indices'][lo:hi], indptr[start:stop + 1] - lo),
                                      shape=(stop - start, n_genes))
            sub = block[want.index.to_numpy() - start]
            ind = sparse.csr_matrix((np.ones(len(want)), (want.to_numpy(), np.arange(len(want)))), shape=(len(lab), len(want)))
            out += (ind @ sub).T.toarray()
    return pd.DataFrame(out, columns=lab)


t0 = time.perf_counter()
row_index = np.flatnonzero(keep.to_numpy())
labels = (PTN.donor_id + '|' + PTN['subclass.l2']).to_numpy()
SUMS_H = h5_group_sums(KPMP, row_index, labels)
SUMS_H.index = HUMAN_GENES
print(f'Pseudobulks {time.perf_counter() - t0:.0f} s; {SUMS_H.shape[1]} donor × subclass columns')
assert np.allclose(SUMS_H.to_numpy(), np.round(SUMS_H.to_numpy())), 'raw counts must be whole numbers'
LIB_H = SUMS_H.sum(axis=0)

## 3 · Human gene statistics

Genes map to mouse through the one-to-one ortholog map. E_h = notebook 94's candidate universes ∩ mapped genes with
whole-PT CPM ≥ 1 in ≥ 3 donors (mt out). Level: limma-trend on whole-PT log2 CPM, AKI against reference. Location: the
same on d = log2 CPM(PT-S3) − log2 CPM(PT-S1) for donors with ≥ 20 nuclei in each.

In [ ]:
orth = pd.read_csv(ORTHOLOGS)
h2m = dict(zip(orth.human_symbol.astype(str), orth.mouse_symbol.astype(str)))
mapped = [g for g in SUMS_H.index if g in h2m]
SUMS = SUMS_H.loc[mapped].groupby(lambda g: h2m[g]).sum()
U71 = pd.read_csv(T71 / 'tables' / 'universe_aki_vs_control.csv').gene.astype(str).tolist()
U76 = pd.read_csv(T76 / 'universe.csv').gene.astype(str).tolist()
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
INDUCED = pd.read_csv(T91 / 'i_induced_genes.csv').gene.astype(str).tolist()
LOST = pd.read_csv(T91 / 'i_lost_genes.csv').gene.astype(str).tolist()
candidates = set(U71) | set(U76) | set(G91.index) | set(INDUCED) | set(LOST)
DONOR_COLS = {d: [c for c in SUMS.columns if c.startswith(d + '|')] for d in DONOR_TABLE.index}
WHOLE = pd.DataFrame({d: SUMS[cols].sum(axis=1) for d, cols in DONOR_COLS.items()})
WHOLE_LIB = pd.Series({d: LIB_H[cols].sum() for d, cols in DONOR_COLS.items()})
whole_cpm = WHOLE / WHOLE_LIB * 1e6
expressed = (whole_cpm >= MIN_CPM).sum(axis=1) >= MIN_DONORS
E = sorted(g for g in candidates if g in WHOLE.index and expressed[g] and not g.startswith(MT_PREFIX))
pd.DataFrame({'gene': E}).to_csv(tables / 'universe.csv', index=False)
print(f'E_h: {len(E):,} genes of {len(candidates):,} candidates ({sum(g in WHOLE.index for g in candidates):,} mapped)')


def limma_frame(values, case, genes, prefix):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fit = limma_trend_species(values, case, genes=genes, trend=True, robust=True)
    return pd.DataFrame({f'{prefix} coefficient': fit.centred, f'{prefix} t': fit.t_centred, f'{prefix} p': fit.p,
                         f'{prefix} q': fit.q}, index=genes)


def contrast_stats(reference):
    donors = DONOR_TABLE.index[DONOR_TABLE.category.isin(['AKI'] + reference)].tolist()
    case = DONOR_TABLE.loc[donors, 'category'].eq('AKI').to_numpy()
    level = limma_frame(log_cpm(WHOLE.loc[E, donors].to_numpy(float), WHOLE_LIB[donors].to_numpy(float)).T, case, E, 'level')
    seg_ok = [d for d in donors if DONOR_TABLE.loc[d, 'PT-S1'] >= MIN_SEGMENT and DONOR_TABLE.loc[d, 'PT-S3'] >= MIN_SEGMENT]
    case_s = DONOR_TABLE.loc[seg_ok, 'category'].eq('AKI').to_numpy()
    s3 = log_cpm(SUMS.loc[E, [f'{d}|PT-S3' for d in seg_ok]].to_numpy(float), LIB_H[[f'{d}|PT-S3' for d in seg_ok]].to_numpy(float))
    s1 = log_cpm(SUMS.loc[E, [f'{d}|PT-S1' for d in seg_ok]].to_numpy(float), LIB_H[[f'{d}|PT-S1' for d in seg_ok]].to_numpy(float))
    location = limma_frame((s3 - s1).T, case_s, E, 'location')
    info = {'donors': len(donors), 'AKI': int(case.sum()), 'location donors': len(seg_ok), 'location AKI': int(case_s.sum())}
    return pd.concat([level, location], axis=1), info


EXT, CONTRAST_INFO = {}, {}
for t in TIMES:
    EXT[t], CONTRAST_INFO[t] = contrast_stats(REFERENCE[t])
EXT = pd.concat(EXT, axis=1)
EXT.to_csv(tables / 'external_genes.csv.gz')
display(pd.DataFrame(CONTRAST_INFO))
display(pd.DataFrame({t: {f'{k} q ≤ 0.05': int((EXT[t][f'{k} q'] <= ALPHA).sum()) for k in ('level', 'location')} for t in TIMES}))

## 4 · Our statistics and the call lists

Internal signs come from notebook 91 for every list: genes' level sign = sign(L), location sign = sign(Δ late − L);
sets' level sign = sign(L), location sign = sign of mean P over the bins centred ≥ 1.8 (91's LATE).

**Guard G4:** each list's call count equals the recorded one.

In [ ]:
S91 = pd.read_csv(T91 / 'i5_library_sets.csv').set_index('set')
late_cols = [c for c in S91.columns if c.startswith('P ') and float(c.split()[1]) >= LATE]
assert [float(c.split()[1]) for c in late_cols] == [1.85, 1.95, 2.05, 2.15, 2.25, 2.35, 2.45]
SET_SIGN = {'level': np.sign(S91.L), 'location': np.sign(S91[late_cols].mean(axis=1))}
GENE_SIGN = {'level': np.sign(G91.L), 'location': np.sign(G91['Δ late'] - G91.L)}

stat71 = pd.read_csv(T71 / 'tables' / 'gene_statistics_aki_vs_control.csv', header=[0, 1], index_col=0)['condition']
stat71.index = stat71.index.astype(str)
cam71 = pd.read_csv(T71 / 'tables' / 'pathway_tests_aki_vs_control_cameraPR_species.csv').set_index('pathway_id')
cam76 = pd.read_csv(T76 / 'pathway_tests_condition.csv').set_index('pathway_id')
calls93 = pd.read_csv(T93 / 'calls_aki_vs_control.csv')
rown = pd.read_csv(T93 / 'row_n_aki_vs_control.csv', header=[0, 1], index_col=0)['condition']
SETS71 = json.loads((T71 / 'tables' / 'gene_sets_aki_vs_control.json').read_text())
SETS76 = json.loads((T76 / 'gene_sets.json').read_text())


def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


clus = stored(T71, 'method_b_aki_vs_control_species__null_default')
clus_genes = pd.DataFrame({c: clus[f'genes|{c}'] for c in clus['genes|__columns'].astype(str)})
clus_genes['gene'] = clus_genes.gene.astype(str)
clus_genes = clus_genes.set_index('gene')
clus_sel = clus_genes[clus_genes.selected.astype(bool) & clus_genes.direction.isin(['up', 'down'])]

usable71 = stat71.converged.astype(bool) & ~stat71.separated.astype(bool) & stat71.q.notna()
LISTS = {
    'G-L91': ('gene', G91.index[G91['level change'].astype(bool)].tolist(), None, list(G91.index)),
    'G-M91': ('gene', G91.index[G91['location change'].astype(bool)].tolist(), None, list(G91.index)),
    'G-F71': ('gene', stat71.index[usable71 & (stat71.q <= ALPHA)].tolist(), None, stat71.index[usable71].tolist()),
    'G-C71': ('gene', clus_sel.index.tolist(), pd.Series(np.where(clus_sel.direction.eq('up'), 1., -1.), index=clus_sel.index),
              clus_genes.index.tolist()),
    'S-F71': ('set', cam71.index[cam71.call.astype(bool)].tolist(), None, cam71.index.tolist()),
    'S-F76': ('set', cam76.index[cam76.call.astype(bool)].tolist(), None, cam76.index.tolist()),
    'S-C71': ('set', calls93.pathway_id[calls93.method.eq('clustering')].tolist(), None, list(SETS71)),
    'S-N93': ('set', calls93.pathway_id[calls93.method.eq('position-split ORA')].tolist(), None, rown.index.astype(str).tolist()),
    'S-L91': ('set', S91.index[S91['level set'].astype(bool)].tolist(), None, S91.index.tolist()),
    'S-M91': ('set', S91.index[S91['location set'].astype(bool)].tolist(), None, S91.index.tolist()),
}
recorded = {'F71': len(LISTS['S-F71'][1]), 'C71': len(LISTS['S-C71'][1]), 'N93': len(LISTS['S-N93'][1]),
            'F76': len(LISTS['S-F76'][1]), 'level genes': len(LISTS['G-L91'][1]), 'location genes': len(LISTS['G-M91'][1]),
            'level sets': len(LISTS['S-L91'][1]), 'location sets': len(LISTS['S-M91'][1])}
assert recorded == FROZEN_CALLS, f'G4: {recorded}'
guard_log.append('G4: the read-only call lists have their recorded sizes (F 103, clustering 459, ORA 134, F76 106, '
                 '91: 3,159 / 362 genes, 538 / 236 sets)')
print('· ' + guard_log[-1])

# programs: 80's published injury programs (Lake 2023's own aPT lists excluded), the claim-check programs
P80 = pd.read_csv(T80 / 'programs.csv').set_index('program_id')
M80 = pd.read_csv(T80 / 'programs_members.csv')
PROGRAMS, PROGRAM_SIGN = {}, {}
for pid, block in M80.groupby('program_id', sort=False):
    if pid.startswith('L23-'):                                  # derived from these KPMP data
        continue
    PROGRAMS[f'80 · {pid}'] = block.gene.astype(str).tolist()
    PROGRAM_SIGN[f'80 · {pid}'] = 1. if P80.loc[pid, 'direction'] == 'up in IR' else -1.
assert set(P80.direction[~P80.index.str.startswith('L23-')]) == {'up in IR', 'down in IR'}
S_P80 = list(PROGRAMS)
SETS76_U = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
PROGRAM_C = sorted((set(SETS76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(SETS76['MSigDB_Hallmark_2020::E2F Targets'])) & SETS76_U)
transporters = pd.read_csv(T76 / 'programs_transporters.csv')
PROGRAM_TPT = sorted(transporters.mouse[transporters.in_T_PT.astype(bool)].astype(str))
assert len(PROGRAM_C) == 297 and len(PROGRAM_TPT) == 93, "76's programs C and T_PT"   # 76's programs.csv truncates members
markers76 = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
assert len(markers76) == 18
features = pd.read_csv(STATE / 'GSM9744085_CTRL_ZT4_01_features.tsv.gz', sep='\t', header=None, names=['id', 'symbol', 'type'])
ens_to_symbol = dict(zip(features.id, features.symbol))
fast = pd.read_csv(STATE / 'GSE267280_RNA_Seq_counts.txt.gz', sep='\t')
fast['symbol'] = fast.Gene_id.str.split('.').str[0].map(ens_to_symbol)
fast = fast.dropna(subset=['symbol']).groupby('symbol')[['Fed_Kidney', 'Fasting_Kidney']].sum()
fcpm = fast / fast.sum() * 1e6
FAST = pd.DataFrame({'lfc': np.log2(fcpm.Fasting_Kidney + 1) - np.log2(fcpm.Fed_Kidney + 1), 'cpm': fcpm.mean(axis=1)})
xi = pd.concat([pd.read_csv(XIONG / f'PT-{s}_fWTvsmWT.csv').assign(segment=s) for s in ('S1', 'S2', 'S3')])
sig = xi[xi.significant.astype(str).str.upper().eq('TRUE')]
CLAIM_PROGRAMS = {
    'K1 · injury markers (76)': (markers76, 'level', 1.),
    'K2 · cell-cycle program C (76)': (PROGRAM_C, 'level', 1.),
    'K3 · transport program T_PT (76)': (PROGRAM_TPT, 'level', -1.),
    'K4 · T_PT, progressive (76)': (PROGRAM_TPT, 'location', -1.),
    'K7 · male-biased PT (Xiong 2023)': (sorted(set(sig.symbol[sig.Direction.eq('DOWN')])), 'level', -1.),
    'K8 · fasting up (GSE267280)': (FAST.index[(FAST.lfc >= FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(), 'level', 1.),
    'K8 · fasting down (GSE267280)': (FAST.index[(FAST.lfc <= -FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(), 'level', -1.),
    'K9 · induced (91)': (INDUCED, 'level', 1.),
    'K9 · lost (91)': (LOST, 'level', -1.),
}
K5 = ['KEGG_2019_Mouse::PPAR signaling pathway', 'KEGG_2019_Mouse::Peroxisome',
      'Reactome_2022::Peroxisomal Lipid Metabolism R-HSA-390918', 'Reactome_2022::Cholesterol Biosynthesis R-HSA-191273',
      'Reactome_2022::Plasma Lipoprotein Assembly R-HSA-8963898', 'MSigDB_Hallmark_2020::Fatty Acid Metabolism']
K6 = ['MSigDB_Hallmark_2020::' + n for n in ('G2-M Checkpoint', 'E2F Targets', 'TNF-alpha Signaling via NF-kB',
                                            'p53 Pathway', 'Apoptosis', 'Inflammatory Response', 'Hypoxia')]
print({k: len(v[0]) for k, v in CLAIM_PROGRAMS.items()})

## 5 · External set statistics

cameraPR (inter-gene correlation 0.01) of each library set, on its members in E (10–300), against the external level
t, location t and |location t|, BH within each contrast. Programs (80's, the claim checks) are
tested in their own family on their members in E (≥ 5).

In [ ]:
from pseudospace.pathway_pipelines import camera_pr

E_SET = set(E)
LIB_SETS = {}
for source in (SETS71, SETS76, {s: [] for s in S91.index}):
    for k, v in source.items():
        LIB_SETS.setdefault(k, set()).update(map(str, v))
LIB_SETS = {k: sorted(v & E_SET) for k, v in LIB_SETS.items()}
LIB_SETS = {k: v for k, v in LIB_SETS.items() if SET_SIZES[0] <= len(v) <= SET_SIZES[1]}
PROG_SETS = {k: sorted(set(v) & E_SET) for k, v in {**PROGRAMS, **{k: v[0] for k, v in CLAIM_PROGRAMS.items()}}.items()}
PROG_SETS = {k: v for k, v in PROG_SETS.items() if len(v) >= MIN_PROGRAM}
print(f'{len(LIB_SETS):,} library sets with 10–300 members in E · {len(PROG_SETS)} programs with ≥ 5')
VECTORS = {'level': ('level t', False), 'location': ('location t', False), '|location|': ('location t', True)}


def camera_table(sets):
    rows = []
    for t in TIMES:
        for name, (column, absolute) in VECTORS.items():
            v = EXT[t][column]
            r = camera_pr(v.abs() if absolute else v, sets, inter_gene_cor=CAMERA_COR).set_index('pathway_id')
            sign = np.where(r.direction.eq('Up'), 1., -1.)
            rows.append(pd.DataFrame({'time': t, 'statistic': name, 'n in E': r.n, 'direction': r.direction,
                                      'z': sign * norm.isf(r.p / 2), 'p': r.p, 'fdr': r.fdr}, index=r.index))
    return pd.concat(rows).rename_axis('set').reset_index()


t0 = time.perf_counter()
CAM = camera_table(LIB_SETS)
CAMP = camera_table(PROG_SETS)
pd.concat([CAM.assign(family='library'), CAMP.assign(family='program')]).to_csv(tables / 'external_sets.csv.gz', index=False)
print(f'External set statistics {time.perf_counter() - t0:.0f} s')
display(CAM[CAM.fdr <= ALPHA].groupby(['statistic', 'time']).size().unstack().reindex(columns=TIMES).rename(columns=T_LABEL))

## 6 · Agreement of levels (descriptive)

Spearman ρ over E_h ∩ notebook 91's genes between our L and the human level coefficient, and between Δ late − L and
the human location coefficient, per contrast. Notebook 94's Kirita values at 12 h sit beside them.

In [ ]:
common = [g for g in E if g in G91.index]
ALIGN = pd.DataFrame({t: {'level ρ': spearmanr(G91.loc[common, 'L'], EXT[t].loc[common, 'level coefficient'])[0],
                          'location ρ': spearmanr(G91.loc[common, 'Δ late'] - G91.loc[common, 'L'],
                                                  EXT[t].loc[common, 'location coefficient'])[0]} for t in TIMES}).T
ALIGN['genes'] = len(common)
ALIGN.to_csv(tables / 'alignment.csv')
display(ALIGN.round(3))

## 7 · Scores

Replicated: external q (genes) or camera FDR (sets) ≤ 0.05 in the item's internal direction (level, location), or
in either direction (unsigned location: genes |location| q; sets the |location t| camera Up). Null: 1,000 random lists
of the same size from the list's pool, matched on expression decile (genes) or size tercile (sets).

In [ ]:
expr91 = G91['log2 CP10k (controls)'].rank(pct=True)
mean_count = pd.Series(np.asarray(clus_genes['expression_per_10k'], float), index=clus_genes.index).rank(pct=True)
EXPR_PCT = expr91.combine_first(mean_count)
EXPR_DECILE = np.minimum((EXPR_PCT * 10).astype(int), 9)
AXES = {'level': 'level', 'location': 'location', 'unsigned location': 'location'}
PRIMARY = {'G-L91': 'level', 'G-M91': 'location', 'G-F71': 'unsigned location', 'G-C71': 'level',
           'S-F71': 'unsigned location', 'S-F76': 'unsigned location', 'S-C71': 'level', 'S-N93': 'location',
           'S-L91': 'level', 'S-M91': 'location', 'S-P80': 'level'}


def gene_hits(t, axis, signs):
    stat = AXES[axis]
    q = EXT[t][f'{stat} q']
    if axis.startswith('unsigned'):
        return q <= ALPHA
    return pd.Series(replicated(EXT[t][f'{stat} coefficient'], q, signs.reindex(E)), index=E)


def set_hits(t, axis, signs, cam):
    stat = {'level': 'level', 'location': 'location', 'unsigned location': '|location|'}[axis]
    block = cam[cam.time.eq(t) & cam.statistic.eq(stat)].set_index('set')
    if axis.startswith('unsigned'):
        return block.direction.eq('Up') & (block.fdr <= ALPHA)
    return pd.Series(replicated(block.z, block.fdr, signs.reindex(block.index)), index=block.index)


rows = []
for name, (kind, items, own_sign, universe) in {**LISTS, 'S-P80': ('set', S_P80, pd.Series(PROGRAM_SIGN), None)}.items():
    for axis in AXES:
        signed = not axis.startswith('unsigned')
        if name == 'S-P80' and axis != 'level':
            continue
        for t in TIMES:
            if kind == 'gene':
                # signs: 91's for the pool; on the level axis a list's own sign replaces 91's for its members
                base = GENE_SIGN['level' if axis == 'level' else 'location'] if signed else pd.Series(1., index=E)
                hits = gene_hits(t, axis, base)
                pool = [g for g in universe if g in hits.index and g in EXPR_DECILE.index and base.get(g, 0) != 0]
                if own_sign is not None and axis == 'level':
                    member_sign = own_sign
                    member_hits = gene_hits(t, axis, own_sign)
                else:
                    member_sign, member_hits = base, hits
                members = [g for g in items if g in hits.index and g in EXPR_DECILE.index and member_sign.get(g, 0) != 0]
                k = int(member_hits.reindex(members).fillna(False).sum())
                strata_pool, strata_list = EXPR_DECILE.reindex(pool).to_numpy(), EXPR_DECILE.reindex(members).to_numpy()
                null = matched_null(hits.reindex(pool).fillna(False).to_numpy(), strata_pool, strata_list, n=N_NULL, seed=SEED)
            else:
                cam = CAMP if name == 'S-P80' else CAM
                signs = SET_SIGN['level' if axis == 'level' else 'location']
                if name == 'S-P80':
                    block = CAMP[CAMP.time.eq(t) & CAMP.statistic.eq('level')].set_index('set')
                    members = [p for p in items if p in block.index]
                    k = int(replicated(block.z.reindex(members), block.fdr.reindex(members), pd.Series(PROGRAM_SIGN).reindex(members)).sum())
                    hits = set_hits(t, axis, signs, CAM)
                    pool = [s for s in S91.index if s in hits.index and signs.get(s, 0) != 0]
                    sizes_pool = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set')['n in E']
                    sizes_list = block['n in E'].reindex(members)
                else:
                    hits = set_hits(t, axis, signs, cam)
                    pool = [s for s in universe if s in hits.index and (not signed or signs.get(s, 0) != 0)]
                    members = [s for s in items if s in set(pool)]
                    k = int(hits.reindex(members).fillna(False).sum())
                    sizes_pool = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set')['n in E']
                    sizes_list = sizes_pool.reindex(members)
                cuts = np.quantile(sizes_pool.reindex(pool).to_numpy(float), [1 / 3, 2 / 3])
                strata_pool = np.digitize(sizes_pool.reindex(pool).to_numpy(float), cuts)
                strata_list = np.digitize(sizes_list.to_numpy(float), cuts)
                null = matched_null(hits.reindex(pool).fillna(False).to_numpy(), strata_pool, strata_list, n=N_NULL, seed=SEED)
            n = len(members)
            rows.append({'list': name, 'kind': kind, 'axis': axis, 'time': t, 'n': n, 'k': k,
                         'precision': k / n if n else np.nan, **null_summary(k, null),
                         'pool': len(pool), 'pool rate': float(hits.reindex(pool).fillna(False).mean()) if pool else np.nan})
SCORES = pd.DataFrame(rows)
SCORES['enrichment'] = SCORES.precision / (SCORES['null mean'] / SCORES.n)
SCORES['null precision high'] = SCORES['null high'] / SCORES.n
SCORES['replicates'] = (SCORES.n >= MIN_N_REPLICATE) & (SCORES.k > SCORES['null high'])
SCORES.to_csv(tables / 'list_scores.csv', index=False)
primary_view = SCORES[SCORES.apply(lambda r: PRIMARY[r['list']] == r['axis'], axis=1)]
display(primary_view.pivot_table(index=['list', 'axis'], columns='time', values='precision').reindex(columns=TIMES)
        .rename(columns=T_LABEL).round(3))
display(primary_view[primary_view.time.eq(T_STAR)][['list', 'axis', 'n', 'k', 'precision', 'null mean', 'null high',
                                                    'p', 'enrichment', 'replicates']].round(3))

### 7.1 · Comparisons and verdicts (rules 1–5)

In [ ]:
COMP = []
for axis in AXES:
    block = SCORES[SCORES.axis.eq(axis)]
    names = sorted(block['list'].unique())
    for a in names:
        for b in names:
            if a == b:
                continue
            row = {'axis': axis, 'A': a, 'B': b}
            beats = {}
            for t in TIMES:
                ra = block[block['list'].eq(a) & block.time.eq(t)].iloc[0]
                rb = block[block['list'].eq(b) & block.time.eq(t)].iloc[0]
                ok = ra.n >= MIN_N_COMPARE and rb.n >= MIN_N_COMPARE
                beats[t] = bool(ok and ra.precision - rb.precision >= MARGIN)
                row[f'Δ precision {T_LABEL[t]}'] = ra.precision - rb.precision if ok else np.nan
            others = [t for t in ACUTE if t != T_STAR]
            row['beats at t*'] = beats[T_STAR]
            row['stable'] = beats[T_STAR] and any(beats[t] for t in others)
            COMP.append(row)
COMP = pd.DataFrame(COMP)
COMP.to_csv(tables / 'comparisons.csv', index=False)

VERDICTS = []
for axis in AXES:
    block = SCORES[SCORES.axis.eq(axis) & SCORES.time.eq(T_STAR) & (SCORES.n >= MIN_N_COMPARE)]
    eligible = block['list'].tolist()
    best = [a for a in eligible if all(COMP[(COMP.axis == axis) & (COMP.A == a) & (COMP.B == b)].stable.iloc[0]
                                       for b in eligible if b != a)]
    VERDICTS.append({'axis': axis, 'lists with n ≥ 20': ', '.join(eligible), 'best': best[0] if best else 'none',
                     'replicating at t*': ', '.join(block['list'][block.replicates])})
star = SCORES[SCORES.time.eq(T_STAR)].set_index(['list', 'axis'])
f_sets = bool(star.loc[('S-F71', 'unsigned location'), 'replicates'])
f_genes = bool(star.loc[('G-F71', 'unsigned location'), 'replicates'])
F_VERDICT = 'replicates' if f_sets and f_genes else 'partly' if (f_sets or f_genes) else 'does not replicate'
VERDICTS = pd.DataFrame(VERDICTS)
display(VERDICTS)
print('Positional F-test, AKI shape calls in human AKI (rule 4):', F_VERDICT, f'(sets {f_sets}, genes {f_genes})')

## 8 · Threshold-free agreement

Spearman ρ over all items between an internal statistic and the external statistic of its axis, per time point.

In [ ]:
f_z = pd.Series(norm.isf(cam71.p_one_sided.astype(float).clip(1e-300, 1 - 1e-16)), index=cam71.index)
AGREE = []
for t in TIMES:
    e = EXT[t]
    gl = [g for g in E if g in G91.index]
    gf = [g for g in E if g in stat71.index and usable71.get(g, False)]
    lvl = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set').z
    loc = CAM[CAM.time.eq(t) & CAM.statistic.eq('location')].set_index('set').z
    aloc = CAM[CAM.time.eq(t) & CAM.statistic.eq('|location|')].set_index('set').z
    s91 = [s for s in S91.index if s in lvl.index]
    sf = [s for s in cam71.index if s in aloc.index]
    pairs = {
        'genes · L vs level': (G91.loc[gl, 'L'], e.loc[gl, 'level coefficient']),
        'genes · Δ late − L vs location': (G91.loc[gl, 'Δ late'] - G91.loc[gl, 'L'], e.loc[gl, 'location coefficient']),
        'genes · F z vs |location t|': (stat71.loc[gf, 'z'].astype(float), e.loc[gf, 'location t'].abs()),
        'sets · 91 z L vs level z': (S91.loc[s91, 'z L'], lvl[s91]),
        'sets · 91 z M × location sign vs location z': (S91.loc[s91, 'z M'] * SET_SIGN['location'][s91], loc[s91]),
        'sets · F z vs |location| z': (f_z[sf], aloc[sf])}
    for k, (a, b) in pairs.items():
        ok = np.isfinite(np.asarray(a, float)) & np.isfinite(np.asarray(b, float))
        AGREE.append({'pair': k, 'time': T_LABEL[t], 'items': int(ok.sum()),
                      'ρ': spearmanr(np.asarray(a, float)[ok], np.asarray(b, float)[ok])[0]})
AGREE = pd.DataFrame(AGREE)
AGREE.to_csv(tables / 'agreement.csv', index=False)
display(AGREE.pivot(index='pair', columns='time', values='ρ')[[T_LABEL[t] for t in TIMES]].round(3))

## 9 · Claim checks (named in the protocol)

In [ ]:
CLAIMS = []
for name, (members, axis, direction) in CLAIM_PROGRAMS.items():
    for t in TIMES:
        stat = 'level' if axis == 'level' else 'location'
        block = CAMP[CAMP.time.eq(t) & CAMP.statistic.eq(stat)].set_index('set')
        r = block.loc[name] if name in block.index else None
        CLAIMS.append({'check': name, 'axis': axis, 'expected': '+' if direction > 0 else '−', 'time': T_LABEL[t],
                       'members in E': int(r['n in E']) if r is not None else 0,
                       'z': float(r.z) if r is not None else np.nan, 'fdr': float(r.fdr) if r is not None else np.nan,
                       'holds': bool(r is not None and r.fdr <= ALPHA and np.sign(r.z) == direction)})
for group, ids, direction in (('K5', K5, -1.), ('K6', K6, 1.)):
    for s in ids:
        for t in TIMES:
            block = CAM[CAM.time.eq(t) & CAM.statistic.eq('location')].set_index('set')
            r = block.loc[s] if s in block.index else None
            CLAIMS.append({'check': f'{group} · {s.split("::", 1)[1]}', 'axis': 'location',
                           'expected': '+' if direction > 0 else '−', 'time': T_LABEL[t],
                           'members in E': int(r['n in E']) if r is not None else 0,
                           'z': float(r.z) if r is not None else np.nan, 'fdr': float(r.fdr) if r is not None else np.nan,
                           'holds': bool(r is not None and r.fdr <= ALPHA and np.sign(r.z) == direction)})
CLAIMS = pd.DataFrame(CLAIMS)
CLAIMS.to_csv(tables / 'claim_checks.csv', index=False)
display(CLAIMS[CLAIMS.time.eq(T_LABEL[T_STAR])].round(3))
display(CLAIMS.pivot(index='check', columns='time', values='holds')[[T_LABEL[t] for t in TIMES]])

### 9.1 · K10: notebook 95's signalling finding in human AKI

PROGENy (native human weights, 500 genes per pathway, mapped to mouse symbols) EGFR, MAPK and PI3K activity: ulm on
the human level t with 1,000 permutations within expression deciles (notebook 95's estimator). Holds when q ≤ 0.05
and z > 0 for each pathway.

In [ ]:
from pseudospace.regulator_activity import progeny_network, stratified_permutations, ulm_null, weight_matrix
from pseudospace.stats_gam import bh_adjust

pg = pd.read_csv(NETS / 'progeny_human.tsv', sep='\t')
FOOT = progeny_network(pg, h2m, top=PROGENY_TOP)
expr_e = G91['log2 CP10k (controls)'].reindex(E)
strata = np.full(len(E), -1)
ok = expr_e.notna().to_numpy()
strata[ok] = np.minimum((expr_e[ok].rank(pct=True).to_numpy() * 10).astype(int), 9)
W, sources = weight_matrix(FOOT, E, min_targets=10)
perms = stratified_permutations(strata, N_PERM, SEED)
K10 = []
for t in TIMES:
    tv, zv = ulm_null(W, EXT[t]['level t'].reindex(E).to_numpy(float), perms)
    q = bh_adjust(2 * norm.sf(np.abs(zv)))
    for s, tt, zz, qq in zip(sources, tv, zv, q):
        K10.append({'contrast': t, 'pathway': s, 't': tt, 'z': zz, 'q': qq})
K10 = pd.DataFrame(K10)
K10['checked'] = K10.pathway.isin(['EGFR', 'MAPK', 'PI3K'])
K10['holds'] = K10.checked & (K10.q <= ALPHA) & (K10.z > 0)
K10.to_csv(tables / 'k10_progeny.csv', index=False)
display(K10.pivot(index='pathway', columns='contrast', values='z').round(2))
K10_HOLDS = bool(K10[K10.checked & K10.contrast.eq(T_STAR)].holds.all())
print('K10 holds (EGFR, MAPK and PI3K up at q ≤ 0.05):', K10_HOLDS)

### 9.2 · Human beside mouse: which calls generalise

In [ ]:
mouse = pd.read_csv(T94 / 'list_scores.csv')
mouse = mouse[mouse.time.eq('12hours')].set_index(['list', 'axis'])
human = SCORES[SCORES.time.eq(T_STAR)].set_index(['list', 'axis'])
CROSS = pd.DataFrame({'human precision': human.precision, 'human enrichment': human.enrichment,
                      'human replicates': human.replicates, 'mouse (Kirita 12 h) precision': mouse.precision.reindex(human.index),
                      'mouse enrichment': mouse.enrichment.reindex(human.index),
                      'mouse replicates': mouse.replicates.reindex(human.index)})
CROSS['primary axis'] = [PRIMARY[l] == a for l, a in CROSS.index]
CROSS.to_csv(tables / 'cross_species.csv')
display(CROSS[CROSS['primary axis']].round(3))
mk = pd.read_csv(T94 / 'claim_checks.csv')
mk = mk[mk.time.eq('12 h')].set_index('check')
hk = CLAIMS[CLAIMS.time.eq(T_STAR)].set_index('check')
CLAIM_CROSS = pd.DataFrame({'human z': hk.z, 'human holds': hk.holds, 'mouse z (12 h)': mk.z.reindex(hk.index),
                            'mouse holds': mk.holds.reindex(hk.index)})
CLAIM_CROSS.to_csv(tables / 'claim_cross_species.csv')
display(CLAIM_CROSS.round(2))

### 9.3 · Post hoc: PT state composition, and identity-retaining nuclei only

Added after the run showed that KPMP's reference donors are dominated by degenerative PT (dPT) nuclei and the AKI
donors by adaptive PT (aPT), so the frozen whole-PT contrast partly compares two injury-state mixtures. Here: the
state shares per group, and the level contrast on identity-retaining nuclei only (PT-S1 + PT-S2 + PT-S3 per donor),
with the gene and set agreement with our L and the level claim checks. Not judged.

In [ ]:
STATES = ['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']
share = DONOR_TABLE[STATES].div(DONOR_TABLE[STATES].sum(axis=1), axis=0)
COMPOSITION = share.groupby(DONOR_TABLE.category).mean()
COMPOSITION.to_csv(tables / 'post_hoc_composition.csv')
display(COMPOSITION.round(3))
donors = DONOR_TABLE.index[DONOR_TABLE.category.isin(['AKI'] + REFERENCE['primary'])].tolist()
ident = {d: [f'{d}|{s}' for s in ('PT-S1', 'PT-S2', 'PT-S3') if f'{d}|{s}' in SUMS.columns] for d in donors}
ID_COUNTS = pd.DataFrame({d: SUMS.loc[E, c].sum(axis=1) for d, c in ident.items()})
ID_LIB = pd.Series({d: LIB_H[c].sum() for d, c in ident.items()})
case = DONOR_TABLE.loc[donors, 'category'].eq('AKI').to_numpy()
ID = limma_frame(log_cpm(ID_COUNTS[donors].to_numpy(float), ID_LIB[donors].to_numpy(float)).T, case, E, 'level')
cam_id = camera_pr(ID['level t'], LIB_SETS, inter_gene_cor=CAMERA_COR).set_index('pathway_id')
z_id = np.where(cam_id.direction.eq('Up'), 1., -1.) * norm.isf(cam_id.p / 2)
z_id = pd.Series(z_id, index=cam_id.index)
s91 = [s for s in S91.index if s in z_id.index]
camp_id = camera_pr(ID['level t'], {k: v for k, v in PROG_SETS.items() if k in CLAIM_PROGRAMS}, inter_gene_cor=CAMERA_COR).set_index('pathway_id')
POST = pd.DataFrame([{
    'level q ≤ 0.05 (genes)': int((ID['level q'] <= ALPHA).sum()),
    'ρ genes L vs level (identity-only)': spearmanr(G91.loc[common, 'L'], ID.loc[common, 'level coefficient'])[0],
    'ρ genes L vs level (whole PT, frozen)': ALIGN.loc[T_STAR, 'level ρ'],
    'ρ sets 91 z L vs level z (identity-only)': spearmanr(S91.loc[s91, 'z L'], z_id[s91])[0]}])
POST.to_csv(tables / 'post_hoc_identity_only.csv', index=False)
display(POST.round(3))
CLAIM_ID = pd.DataFrame({'z': np.where(camp_id.direction.eq('Up'), 1., -1.) * norm.isf(camp_id.p / 2), 'fdr': camp_id.fdr},
                        index=camp_id.index)
CLAIM_ID['expected'] = [CLAIM_PROGRAMS[k][2] for k in CLAIM_ID.index]
CLAIM_ID['holds'] = (CLAIM_ID.fdr <= ALPHA) & (np.sign(CLAIM_ID.z) == CLAIM_ID.expected)
CLAIM_ID = CLAIM_ID[[CLAIM_PROGRAMS[k][1] == 'level' for k in CLAIM_ID.index]]
CLAIM_ID.to_csv(tables / 'post_hoc_identity_only_claims.csv')
display(CLAIM_ID.round(3))

## 10 · Figures

- 98a: each list on its primary axis: human precision against its matched null.
- 98b: human against mouse (Kirita 12 h) enrichment per list.
- 98c: claim checks, human and mouse z side by side.

In [ ]:
order = ['G-L91', 'G-C71', 'S-L91', 'S-C71', 'S-P80', 'G-M91', 'S-M91', 'S-N93', 'G-F71', 'S-F71', 'S-F76']
fig, ax = plt.subplots(figsize=(80 * MM, 62 * MM))
pv = primary_view[primary_view.time.eq(T_STAR)].set_index('list').reindex(order)
y = np.arange(len(order))[::-1]
for yi, (name, r) in zip(y, pv.iterrows()):
    ax.plot([r['null low'] / r.n, r['null high'] / r.n], [yi, yi], color=LIGHT, lw=3, solid_capstyle='butt')
    ax.plot(r['null mean'] / r.n, yi, '|', color=MUTED, ms=6)
    ax.plot(r.precision, yi, 'o', color=LIST_COLOR[name], ms=4)
ax.set_yticks(y, [f'{n} · {PRIMARY[n]}' for n in order])
ax.set_xlabel('share replicated in human AKI')
ax.set_xlim(0, 1)
save(fig, '98a_list_scores_human')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 62 * MM))
cv = CROSS[CROSS['primary axis']].reset_index()
for _, r in cv.iterrows():
    ax.scatter(r['mouse enrichment'], r['human enrichment'], s=16, color=LIST_COLOR[r['list']],
               marker='o' if r['list'].startswith('S-') else 's')
    ax.annotate(r['list'], (r['mouse enrichment'], r['human enrichment']), fontsize=5, xytext=(2, 2), textcoords='offset points')
ax.axhline(1, color=MUTED, lw=.5); ax.axvline(1, color=MUTED, lw=.5)
ax.set_xlabel('enrichment over matched null, mouse IRI (12 h)')
ax.set_ylabel('enrichment over matched null, human AKI')
save(fig, '98b_human_vs_mouse')
plt.show()

cc = CLAIM_CROSS.copy()
fig, ax = plt.subplots(figsize=(85 * MM, 4 * MM * len(cc) + 12 * MM))
yy = np.arange(len(cc))
ax.barh(yy - .2, cc['mouse z (12 h)'], height=.38, color='#0072B2', label='mouse IRI 12 h')
ax.barh(yy + .2, cc['human z'], height=.38, color='#D55E00', label='human AKI')
for i, (_, r) in enumerate(cc.iterrows()):
    if r['human holds']:
        ax.plot(r['human z'], i + .2, 'o', color=INK, ms=2)
    if r['mouse holds']:
        ax.plot(r['mouse z (12 h)'], i - .2, 'o', color=INK, ms=2)
ax.set_yticks(yy, cc.index)
ax.invert_yaxis()
ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('external camera z')
ax.legend(loc='lower right')
save(fig, '98c_claims_human_mouse')
plt.show()

## 11 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', "Notebook 94's scoring code, unchanged except: two contrasts replace the time points; no assigned-nuclei axes; S-P80 drops Lake 2023's L23 lists and keeps Kirita's."),
    ('reading', 'Human genes map to mouse through the one-to-one ortholog map; duplicated human symbols are summed.'),
    ('reading', "Internal signs, expression strata and the clustering, F-test and 91 lists are read exactly as in notebook 94."),
    ('reading', "K10 permutation strata use 91's control expression deciles over E_h (genes without it form one stratum)."),
    ('post hoc', '9.3: PT state composition by group, and the level contrast on identity-retaining nuclei (PT-S1/S2/S3) only. Not judged.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '98_pt_human_aki_benchmark', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'contrasts': CONTRAST_INFO, 'universe E_h': len(E), 'library sets': len(LIB_SETS), 'programs': len(PROG_SETS),
       'F-test verdict (rule 4)': F_VERDICT, 'best per axis': dict(zip(VERDICTS.axis, VERDICTS.best)), 'K10 holds': K10_HOLDS,
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))